In [ ]:
import pandas as pd
import geopandas as gpd
import folium

# ============================================================
# 0. LOAD & CLEAN DEPARTMENTS (CRITICAL)
# ============================================================

departments = pd.read_csv("departments.csv")

# Drop deleted / placeholder rows
departments = departments[
    (departments["DepartmentKey"] != -3) &
    (departments["CensusTract"].notna())
]

# Clean CensusTract
departments["CensusTract"] = (
    departments["CensusTract"]
    .astype(str)
    .str.replace(".0", "", regex=False)
)

# Keep only valid 11-digit tracts
departments = departments[
    departments["CensusTract"].str.fullmatch(r"\d{11}")
]

# Derive CountyFIPS from tract
departments["CountyFIPS"] = departments["CensusTract"].str[:5]
departments["CountyFIPS"] = departments["CountyFIPS"].astype(str).str.zfill(5)

# ✅ AUTHORITATIVE COUNT (value_counts as requested)
dept_counts_by_county = (
    departments["CountyFIPS"]
    .value_counts()
    .reset_index()
)

dept_counts_by_county.columns = ["CountyFIPS", "dept_count"]

# ============================================================
# 1. LOAD COUNTY GEOMETRY (TIGER)
# ============================================================

counties = gpd.read_file("tl_2025_us_county.shp")

# Build CountyFIPS from STATEFP + COUNTYFP
counties["CountyFIPS"] = (
    counties["STATEFP"].astype(str).str.zfill(2) +
    counties["COUNTYFP"].astype(str).str.zfill(3)
)

# Filter to Kansas (20)
counties = counties[counties["STATEFP"] == "20"]

# Use county name
if "NAME" in counties.columns:
    counties["County"] = counties["NAME"]
else:
    counties["County"] = counties["NAMELSAD"]

counties_web = counties.to_crs(epsg=4326)

# ============================================================
# 2. MERGE DEPARTMENT COUNTS INTO COUNTIES (MAP FEED)
# ============================================================

counties_web = counties_web.merge(
    dept_counts_by_county,
    on="CountyFIPS",
    how="left"
)

counties_web["dept_count"] = counties_web["dept_count"].fillna(0).astype(int)

# ============================================================
# 3. COUNTY ACCESS CATEGORIZATION (COUNTS)
# ============================================================

def dept_category(n):
    if n == 0:
        return "0"
    elif n == 1:
        return "1"
    elif n <= 3:
        return "2-3"
    elif n <= 10:
        return "4-10"
    elif n <= 50:
        return "10-50"
    elif n <= 100:
        return "50-100"
    else:
        return "100+"

counties_web["dept_cat"] = counties_web["dept_count"].apply(dept_category)

dept_color_map = {
    "0":       "#CAE5E7",
    "1":       "#AAD6D9",
    "2-3":     "#89C7CB",
    "4-10":    "#68B7BE",
    "10-50":   "#47A8B0",
    "50-100":  "#2799A2",
    "100+":    "#03767E"
}

# ============================================================
# 4. CREATE MAP (NO BASEMAP LABEL)
# ============================================================

m = folium.Map(
    location=[
        counties_web.geometry.centroid.y.mean(),
        counties_web.geometry.centroid.x.mean()
    ],
    zoom_start=6,
    tiles=None
)

folium.TileLayer(
    tiles="cartodbpositron",
    name="",
    control=False
).add_to(m)

# ============================================================
# 5. COUNTY DEPARTMENT COUNTS (FOREGROUND)
# ============================================================

county_count_layer = folium.FeatureGroup(
    name="Department Count (Counties)",
    overlay=False,
    show=True
)

folium.GeoJson(
    counties_web,
    style_function=lambda f: {
        "fillColor": dept_color_map[f["properties"]["dept_cat"]],
        "color": "gray",
        "weight": 0.8,
        "fillOpacity": 0.85,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["County", "dept_count"],
        aliases=["County:", "Number of Departments:"],
        sticky=True
    )
).add_to(county_count_layer)

county_count_layer.add_to(m)

# ============================================================
# 6. BASE LAYER: PATIENTS BY COUNTY
# ============================================================

patients = pd.read_csv("patients.csv")

patients = patients[
    patients["CensusBlockGroupFipsCode"].notna() &
    (patients["CensusBlockGroupFipsCode"] != "*Unspecified")
]

patients["CensusBlockGroupFipsCode"] = (
    patients["CensusBlockGroupFipsCode"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.zfill(12)
)

patients["CountyFIPS"] = patients["CensusBlockGroupFipsCode"].str[:5]

patient_counts = (
    patients
    .groupby("CountyFIPS", as_index=False)
    .size()
    .rename(columns={"size": "patient_count"})
)

counties_web = counties_web.merge(
    patient_counts,
    on="CountyFIPS",
    how="left"
)

counties_web["patient_count"] = counties_web["patient_count"].fillna(0).astype(int)

# Bin patient volume for visualization
def patient_bin(n):
    if n == 0:
        return "No patients"
    elif n < 500:
        return "<500"
    elif n < 2000:
        return "500–2k"
    elif n < 5000:
        return "2k–5k"
    else:
        return "5k+"

counties_web["patient_cat"] = counties_web["patient_count"].apply(patient_bin)

patient_colors = {
    "No patients": "#DDDDDF",
    "<500": "#BBBBBE",
    "500–2k": "#AAAAAE",
    "2k–5k": "#7F8086",
    "5k+": "#54555D"
}

layer_patients = folium.FeatureGroup(
    name="Patients Living in County",
    overlay=False,
    show=False
)

folium.GeoJson(
    counties_web,
    style_function=lambda f: {
        "fillColor": patient_colors[f["properties"]["patient_cat"]],
        "color": "gray",
        "weight": 0.6,
        "fillOpacity": 0.8,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["County", "patient_count"],
        aliases=["County:", "Patients living here:"],
        sticky=True
    )
).add_to(layer_patients)

layer_patients.add_to(m)

# ============================================================
# 7. ACCESS EQUITY TOGGLE (RELATIVE TO POPULATION)
# ============================================================

# Load census population
census = pd.read_csv("tigercensuscodes.csv")
census["GEOID"] = census["GEOID"].astype(str)
census["CountyFIPS"] = census["GEOID"].str[:5].astype(str).str.zfill(5)

county_population = (
    census
    .groupby("CountyFIPS", as_index=False)["PopulationValue"]
    .sum()
    .rename(columns={"PopulationValue": "population"})
)

counties_web = counties_web.merge(
    county_population,
    on="CountyFIPS",
    how="left"
)

counties_web["population"] = counties_web["population"].fillna(0)

# Access ratio
counties_web["depts_per_capita"] = (
    counties_web["dept_count"] / counties_web["population"]
).fillna(0)

state_avg = counties_web["depts_per_capita"].mean()

counties_web["access_ratio"] = (
    counties_web["depts_per_capita"] / state_avg
).replace([float("inf")], 0)

def access_ratio_bin(r):
    if r == 0:
        return "No access"
    elif r < 0.1:
        return "Severely underserved"
    elif r < 0.5:
        return "Underserved"
    elif r < 4.0:
        return "Proportional"
    elif r < 30.0:
        return "Well served"
    else:
        return "Over-served"

counties_web["access_ratio_cat"] = counties_web["access_ratio"].apply(access_ratio_bin)

access_ratio_colors = {
    "No access":            "#F0AF20",
    "Severely underserved": "#F5CA6A",
    "Underserved":          "#FAE4B5",
    "Proportional":         "#F0F0F0",
    "Well served":          "#C8A2B4",
    "Over-served":          "#904469"
}

access_equity_layer = folium.FeatureGroup(
    name="Access Equity (Departments vs Population)",
    overlay=False,
    show=False
)

folium.GeoJson(
    counties_web,
    style_function=lambda f: {
        "fillColor": access_ratio_colors[f["properties"]["access_ratio_cat"]],
        "color": "gray",
        "weight": 0.6,
        "fillOpacity": 0.8,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["County", "dept_count", "population", "access_ratio"],
        aliases=[
            "County:",
            "Departments:",
            "Population:",
            "Access ratio:"
        ],
        sticky=True
    )
).add_to(access_equity_layer)

access_equity_layer.add_to(m)

# ============================================================
# 8. LEGENDS
# ============================================================

legend_count_html = """
<div style="
    position: fixed;
    bottom: 30px;
    left: 30px;
    width: 280px;
    background-color: white;
    border: 2px solid #444;
    z-index: 5000;
    font-size: 14px;
    padding: 10px;
">
<b>Number of Departments per County</b><br>
<i style="background:#CAE5E7;width:18px;height:18px;display:inline-block"></i> 0<br>
<i style="background:#AAD6D9;width:18px;height:18px;display:inline-block"></i> 1<br>
<i style="background:#89C7CB;width:18px;height:18px;display:inline-block"></i> 2–3<br>
<i style="background:#68B7BE;width:18px;height:18px;display:inline-block"></i> 4–10<br>
<i style="background:#47A8B0;width:18px;height:18px;display:inline-block"></i> 10–50<br>
<i style="background:#2799A2;width:18px;height:18px;display:inline-block"></i> 50–100<br>
<i style="background:#03767E;width:18px;height:18px;display:inline-block"></i> 100+<br>
</div>
"""

legend_ratio_html = """
<div style="
    position: fixed;
    bottom: 30px;
    right: 30px;
    width: 320px;
    background-color: white;
    border: 2px solid #444;
    z-index: 5000;
    font-size: 14px;
    padding: 10px;
">
<b>Access Equity (Departments vs Population)</b><br>
<i style="background:#F0AF20;width:18px;height:18px;display:inline-block"></i> No access<br>
<i style="background:#F5CA6A;width:18px;height:18px;display:inline-block"></i> Severely underserved<br>
<i style="background:#FAE4B5;width:18px;height:18px;display:inline-block"></i> Underserved<br>
<i style="background:#F0F0F0;width:18px;height:18px;display:inline-block;border:1px solid #999"></i> Proportional<br>
<i style="background:#C8A2B4;width:18px;height:18px;display:inline-block"></i> Well served<br>
<i style="background:#904469;width:18px;height:18px;display:inline-block"></i> Over-served<br>
</div>
"""


legend_patients_html = """
<div style="
    position: fixed;
    top: 30px;
    left: 30px;
    width: 260px;
    background-color: white;
    border: 2px solid #444;
    z-index: 5000;
    font-size: 14px;
    padding: 10px;
">
<b>Patients Living in County</b><br>

<i style="background:#DDDDDF;width:18px;height:18px;display:inline-block;border:1px solid #999"></i> No patients<br>
<i style="background:#BBBBBE;width:18px;height:18px;display:inline-block"></i> &lt;500<br>
<i style="background:#98999E;width:18px;height:18px;display:inline-block"></i> 500–2k<br>
<i style="background:#76777D;width:18px;height:18px;display:inline-block"></i> 2k–5k<br>
<i style="background:#54555D;width:18px;height:18px;display:inline-block"></i> 5k+<br>

</div>
"""

title_html = """
<div style="
    position: fixed;
    top: 10px;
    left: 50%;
    transform: translateX(-50%);
    z-index: 6000;
    background-color: white;
    padding: 10px 20px;
    border: 2px solid #444;
    border-radius: 6px;
    font-size: 20px;
    font-weight: bold;
    box-shadow: 2px 2px 5px rgba(0,0,0,0.3);
">
SVH Healthcare Access & Patient Distribution
</div>
"""

m.get_root().html.add_child(folium.Element(title_html))

m.get_root().html.add_child(folium.Element(legend_count_html))
m.get_root().html.add_child(folium.Element(legend_ratio_html))
m.get_root().html.add_child(folium.Element(legend_patients_html))


# ============================================================
# 9. LAYER CONTROL + SAVE
# ============================================================

folium.LayerControl(collapsed=False, position="topright").add_to(m)

m.save("kansas_access_map_FINAL_MAP.html")